# PET coincidence explorer

Choose **Runtime → Run all** in Colab, then use the controls. Two tracer deposits generate simulated annihilations inside a detector ring. Each event produces two nearly opposite photons. A detector pair in coincidence gives a **line of response (LOR)**.

The two images use the same events. One spreads each event along its line. The other uses its measured arrival-time difference to better localize the annihilation.

## What timing adds

Measure position $s$ from an LOR’s midpoint, positive toward detector B. If B detects its photon first, then $\Delta t=t_A-t_B$ is positive and the timing estimate is closer to B.

$$\hat s=\frac{c\,\Delta t}{2},\qquad
\mathrm{FWHM}_s=\frac{c\,\mathrm{FWHM}_{\Delta t}}{2},
\qquad c\approx0.300~\mathrm{mm/ps}.$$

FWHM means **full width at half maximum**. Here it describes the uncertainty along one line, in mm. The timing resolution control changes the uncertainty of the measured time difference. The event count control changes how many detector pairs enter both images.

In [ ]:
#@title Launch PET explorer
from functools import lru_cache
from io import BytesIO

import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import Normalize
from IPython.display import display

try:
    from google.colab import output
except ImportError:
    pass
else:
    output.enable_custom_widget_manager()

C = 0.299792458  # mm/ps
RING_RADIUS = 105  # mm
BODY_RADIUS = 80  # mm, displayed object boundary
CENTERS = np.array([[-28, -12], [27, 18]])
GRID = np.linspace(-80, 80, 161)
X, Y = np.meshgrid(GRID, GRID)
inside = X**2 + Y**2 <= BODY_RADIUS**2
rng = np.random.default_rng(313)
n_available = 400
events = CENTERS[rng.integers(0, len(CENTERS), n_available)] + rng.normal(
    scale=4, size=(n_available, 2))
angles = rng.uniform(0, 2*np.pi, n_available)
directions = np.column_stack((np.cos(angles), np.sin(angles)))
normals = np.column_stack((-np.sin(angles), np.cos(angles)))
offsets = (events * normals).sum(axis=1)
midpoints = offsets[:, None] * normals
half_chords = np.sqrt(RING_RADIUS**2 - offsets**2)
end_a = midpoints - half_chords[:, None] * directions
end_b = midpoints + half_chords[:, None] * directions
true_s = (events * directions).sum(axis=1)
timing_errors = rng.standard_normal(n_available)
FWHM_TO_SIGMA = 2*np.sqrt(2*np.log(2))


@lru_cache(maxsize=12)
def backprojections(count, timing_fwhm):
    """Sum unit-weight line kernels with and without TOF information."""
    sigma_s = C * timing_fwhm / (2 * FWHM_TO_SIGMA)
    estimated_s = true_s + sigma_s * timing_errors
    line_map, tof_map = np.zeros_like(X), np.zeros_like(X)
    for i in range(count):
        across = X*normals[i, 0] + Y*normals[i, 1] - offsets[i]
        along = X*directions[i, 0] + Y*directions[i, 1]
        line = np.exp(-0.5*(across/2)**2) * inside
        tof = line * np.exp(-0.5*((along-estimated_s[i])/sigma_s)**2)
        # Every accepted pair contributes the same total weight in each image.
        line_map += line / line.sum()
        tof_map += tof / tof.sum()
    return line_map/count, tof_map/count


def dashboard(count, timing_fwhm):
    line_map, tof_map = backprojections(count, timing_fwhm)
    fig, axes = plt.subplots(1, 3, figsize=(11.6, 3.9), layout="constrained")
    ax = axes[0]
    ax.add_patch(plt.Circle((0, 0), RING_RADIUS, fill=False, color="#536473"))
    ax.add_patch(plt.Circle((0, 0), BODY_RADIUS, fill=False, ls="--", ec="0.6"))
    for i in range(min(count, 12)):
        ax.plot([end_a[i, 0], end_b[i, 0]], [end_a[i, 1], end_b[i, 1]],
                lw=1, color="#157a80", alpha=.55)
    ax.scatter(CENTERS[:, 0], CENTERS[:, 1], marker="x", s=55,
               color="#bb4738", label="Deposit centers")
    ax.set(title=f"First {min(count, 12)} lines", xlim=(-112, 112),
           ylim=(-112, 112), aspect="equal", xlabel="x (mm)", ylabel="y (mm)")
    ax.legend(loc="lower center", fontsize=8)
    # Both images share one scale at each setting.
    norm = Normalize(0, max(line_map.max(), tof_map.max()))
    for ax, values, title in zip(axes[1:], (line_map, tof_map),
                                 ("Without timing", "With time of flight")):
        im = ax.imshow(values, extent=(-80.5, 80.5, -80.5, 80.5),
                       origin="lower", cmap="magma", norm=norm)
        ax.set(title=title, xlabel="x (mm)", ylabel="y (mm)")
    fig.colorbar(im, ax=axes[1:], shrink=.68, label="Relative weight per pixel")
    return fig


event_count = widgets.SelectionSlider(
    options=[8, 25, 80, 200, 400], value=80, description="Detected pairs",
    continuous_update=False, style={"description_width": "initial"},
    layout=widgets.Layout(width="360px", max_width="100%"))
timing = widgets.IntSlider(
    min=150, max=750, step=50, value=400, description="Timing FWHM (ps)",
    continuous_update=False, style={"description_width": "initial"},
    layout=widgets.Layout(width="380px", max_width="100%"))
readout = widgets.HTML()
plot = widgets.Image(format="png", layout=widgets.Layout(width="100%", max_width="1100px"))


def refresh(change=None):
    count, fwhm = event_count.value, timing.value
    fig = dashboard(count, fwhm)
    try:
        buffer = BytesIO()
        fig.savefig(buffer, format="png", dpi=115, facecolor="white")
        plot.value = buffer.getvalue()
    finally:
        plt.close(fig)
    readout.value = (
        f"<b>{count} detected pairs.</b> Along-line uncertainty: "
        f"<b>{C*fwhm/2:.1f} mm FWHM</b> at {fwhm} ps timing FWHM."
    )


event_count.observe(refresh, names="value")
timing.observe(refresh, names="value")
refresh()
display(widgets.VBox([
    widgets.Box([event_count, timing], layout=widgets.Layout(flex_flow="row wrap")),
    readout, plot
]))

## Investigate and explain

1. **One detector pair.** Set **Detected pairs**, the number of measured annihilations, to 8. Pick a line. How much does the pair tell you about where the annihilation occurred?
2. **More pairs.** Keep the timing resolution at 400 ps and increase from 8 to 400 pairs. Which features become easier to recognize? Which extra lines or streaks remain?
3. **Better photon timing.** Keep the pair count fixed and change from 600 to 200 ps. Use the readout to find the along-line FWHM in each case. How does the image change?

The images show sums of backprojected event weights. Each event contributes equal total weight, with the timing being used to redistribute the weight along the line. Both panels share a color scale at each setting. The scale auto-adjusts based on the controls. Compare feature shapes and separation across settings. The detector ring and events are idealized. The model assumes straight rays, continuous detector positions, and Gaussian timing errors, and omits attenuation, scatter, random pairs, and positron range.

[Further reading: Stanford Molecular Imaging Instrumentation Laboratory, TOF PET](https://med.stanford.edu/miil/research/tofdetector.html)